# Comparing Terraform vs OpenTofu: syntax, providers, state, and migration path

A hands-on comparison of the two IaC CLIs against identical configuration — what carries over unchanged, where they differ, and one way to move a small project from one to the other.

## Purpose

Terraform and OpenTofu both consume the same HCL configuration language and expose the same `init` / `plan` / `apply` / `destroy` workflow. This notebook runs one shared `local_file` configuration under each binary (no cloud credentials needed), then walks through provider resolution, state backends, and a migration path step by step.

This is one way to compare them; the docs also suggest deeper dives per provider — what follows is the smallest experiment that still shows the real similarities and differences.

## Setup

Both CLIs need to be on `PATH`. Each experiment runs in its own empty scratch directory so the two state files never mix.

In [ ]:
# last_verified: 2026-09-16 · Terraform/OpenTofu comparison
# Setup: one scratch directory per tool so state files never mix
import json
import os
import subprocess
from pathlib import Path

tf_dir = Path("/tmp/tf-compare-terraform")
ot_dir = Path("/tmp/tf-compare-opentofu")
for d in (tf_dir, ot_dir):
    d.mkdir(parents=True, exist_ok=True)
print(f"terraform dir: {tf_dir}")
print(f"opentofu dir: {ot_dir}")

In [ ]:
# Confirm both binaries respond (informational — the cells below assume they exist)
for binary in ("terraform", "tofu"):
    result = subprocess.run([binary, "version"], capture_output=True, text=True)
    first_line = result.stdout.strip().splitlines()[0] if result.stdout.strip() else "not found"
    print(f"{binary}: {first_line}")

## Step 1 — the same HCL runs on both

For basic resource configuration the syntax is interchangeable: `terraform {}` blocks, `variable` blocks, resource blocks, and outputs are spelled the same way. Write once, run under either binary.

In [ ]:
# One shared configuration, written to both scratch directories.
# No version constraint is pinned here on purpose — the point is the shared shape,
# not any particular provider release.
shared_main = '''
terraform {
  required_providers {
    local = {
      source = "hashicorp/local"
    }
  }
}

provider "local" {}

variable "greeting" {
  type    = string
  default = "hello from shared HCL"
}

resource "local_file" "greeting" {
  content  = var.greeting
  filename = "${path.module}/greeting.txt"
}

output "greeting_path" {
  value = local_file.greeting.filename
}
'''

for d in (tf_dir, ot_dir):
    with open(d / "main.tf", "w") as f:
        f.write(shared_main)
print("Wrote identical main.tf to both directories")

In [ ]:
# Run the shared config under Terraform
def run(cmd, cwd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    tail = (result.stdout or result.stderr).strip().splitlines()[-3:]
    print(f"$ {' '.join(cmd)}  (rc={result.returncode})")
    print("\n".join(tail))
    return result

run(["terraform", "init"], tf_dir)
run(["terraform", "apply", "-auto-approve"], tf_dir)
print("file exists:", (tf_dir / "greeting.txt").is_file())

In [ ]:
# Run the exact same config under OpenTofu — same verbs, same flags
run(["tofu", "init"], ot_dir)
run(["tofu", "apply", "-auto-approve"], ot_dir)
print("file exists:", (ot_dir / "greeting.txt").is_file())

### What this shows about syntax

- The identical `main.tf` plans and applies under both binaries with no edits.
- The CLI verbs and flags (`init`, `plan`, `apply -auto-approve`, `destroy -auto-approve`) match one-to-one for this workflow.
- Plan/apply output reads the same way — adds, changes, and destroys are reported in the same style, so existing run-book habits transfer.

## Step 2 — providers

Provider *requirements* look the same (`required_providers` with a `source` address), but each tool resolves them through its own default registry hostname. In practice the common namespaces are mirrored, so `source = "hashicorp/local"` works under both — and the cell above proves it for this provider.

Two things worth verifying in your own setup rather than assuming: a lock file generated by one binary records that binary's registry hashes, so expect the other binary to re-resolve providers on its first `init`; and community or private providers should be checked per provider instead of assumed mirrored.

## Step 3 — state backends

Both tools default to a local `terraform.tfstate` JSON file in the working directory, and both configure remote state with a `backend` block inside the `terraform {}` block using the same shape. The snippet below is written out for inspection only — it is not applied, since it needs real bucket credentials.

In [ ]:
# Backend configuration shape is identical for both tools (shown, not applied).
# Swap in real bucket/key values before using it for anything real.
backend_example = '''
terraform {
  backend "s3" {
    bucket = "example-state-bucket"
    key    = "demo/terraform.tfstate"
    region = "us-east-1"
  }
}
'''

with open(tf_dir / "backend.example.tf", "w") as f:
    f.write(backend_example)
print(backend_example)

In [ ]:
# Compare the state files both runs produced (skips any side whose binary was missing)
for label, d in (("terraform", tf_dir), ("opentofu", ot_dir)):
    state_file = d / "terraform.tfstate"
    if not state_file.is_file():
        print(f"{label}: no state file — binary may be missing, nothing to compare")
        continue
    state = json.loads(state_file.read_text())
    addresses = [r.get("address") for r in state.get("resources", [])]
    print(f"{label}: top-level keys={sorted(state.keys())} resources={addresses}")

### What this shows about state

- Local state lives at the same path (`terraform.tfstate`) with the same JSON layout — top-level keys and the `resources` list line up between the two runs.
- Remote-backend configuration uses the same block shape, so moving the backend stanza across is a copy, not a rewrite.
- State is per working directory either way, which is what makes the copy-and-replan migration in the next step possible.

## Step 4 — one migration path

This is one way to move a small project; I have only tried the copy-and-replan route, and it avoids touching the original until the new side proves itself:

1. Copy the whole working directory (configuration plus state) to a new folder as a fallback.
2. In the copy, run `tofu init` so providers resolve through the new registry.
3. Run `tofu plan` and expect no changes — an empty plan means the new binary reads the existing state the same way.
4. Continue operating with `tofu`; keep the original directory around until a few clean applies have passed, then retire it.

## Head-to-head

| Aspect | Terraform | OpenTofu |
|---|---|---|
| Config language | HCL (`terraform`, `variable`, resource, output blocks) | Same HCL — shared config applies under both |
| Daily CLI verbs | `init` / `plan` / `apply` / `destroy` with the same flags | Same verbs and flags for this workflow |
| Provider requirements | `required_providers` with `source` addresses | Same block shape; resolves via its own default registry |
| Lock files | Records its registry hashes on first `init` | Same mechanism; expect a re-resolve when switching sides |
| Local state | `terraform.tfstate` JSON in the working dir | Same path, same JSON layout |
| Remote backends | `backend` block inside `terraform {}` | Same block shape — a copy, not a rewrite |
| Migration effort | Starting point | Copy dir, `init`, empty-plan check, cut over |

### When to reach for which

- **Stay where the team already is** when providers, modules, and run-books are settled — the switching cost is low but still nonzero (re-resolve, re-verify).
- **Try the other side on a copy first** when evaluating — the copy-and-replan loop above gives an empty-plan signal before anything real changes.
- **Check community/private providers individually** before committing to a move — the common namespaces carry over, but anything unusual deserves its own trial.

## Verify

To confirm the comparison on your own machine:

1. Run `terraform plan` and `tofu plan` against the identical `main.tf` — both should propose the same single `local_file` add.
2. After applying on both sides, run `terraform state list` and `tofu state list` — expect the same resource addresses.
3. For the migration path, copy one side's directory, run `tofu init` then `tofu plan` in the copy — expect an empty plan before cutting over.
4. Compare the two `terraform.tfstate` files' `resources` arrays — same addresses and attributes means both sides see the same world.

## What I'd try next

Run the same comparison with a remote backend and with a community provider to see where the symmetry breaks — the local-only experiment above is the easy case, and the interesting differences will show up once real state locking and third-party providers are involved.

In [ ]:
# Clean up both scratch runs
run(["terraform", "destroy", "-auto-approve"], tf_dir)
run(["tofu", "destroy", "-auto-approve"], ot_dir)